# Sinmate: QLoRA Fine-Tuning Gemma-3-1B-IT for Spoken Sinhala

This Google Colab notebook fine-tunes **`google/gemma-3-1b-it`** using **4-bit QLoRA** on a conversational Sinhala dataset (*Kathitha Bhashawa* with Sri Lankan fillers, Singlish, and 4 personas).

### Pipeline Overview:
1. **Load Base Model**: `google/gemma-3-1b-it` quantized to 4-bit (VRAM footprint ~1.8GB on free Colab T4).
2. **Chat Template**: Standard Gemma turn tags `<start_of_turn>user...<end_of_turn>`.
3. **Train with QLoRA**: LoRA adapters on linear attention and MLP layers (`r=16, alpha=32`).
4. **Inference Test**: Multi-persona conversational test in colloquial Sinhala.
5. **Export GGUF**: Export 4-bit quantized GGUF weights (~700MB–800MB) for offline laptop execution.

In [9]:
# 1. Install fine-tuning dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece "torchao>=0.16.0"


In [10]:
# 2. Hugging Face Login (Access Gemma license)
# Token lookup order: Colab secret -> HF_TOKEN env var -> getpass prompt
import os
from getpass import getpass
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass  # not on Colab, or no HF_TOKEN secret / notebook access

token = token or os.environ.get('HF_TOKEN') or getpass('Enter your Hugging Face access token: ')
login(token=token)

In [11]:
# 3. Clone or upload Sinmate dataset
import os
if not os.path.exists("data"):
    !git clone https://github.com/chathuwa-whiz/Sinmate.git
    %cd Sinmate

from datasets import load_dataset
dataset = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
print("Dataset loaded:", dataset)

Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 106
    })
    val: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 11
    })
})


In [12]:
# 4. Load Gemma-3-1B-IT in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-1b-it"

# bf16 needs compute capability >= 8.0 (A100/L4/...). The free-tier T4 is 7.5: bf16 kernels fail there,
# and Gemma 3 overflows in fp16, so fall back to fp32 compute (fine for a 1B model).
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print(f"GPU: {torch.cuda.get_device_name()} | compute dtype: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

GPU: Tesla T4 | compute dtype: torch.float32


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [13]:
# 5. Format Chat Templates & Train with Response-Only Loss Masking
import math
import re
from trl import SFTConfig, SFTTrainer

# Ensure tokenizer padding token is defined for batched training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Use conversational messages format directly (TRL applies chat template automatically)
train_dataset = dataset["train"].select_columns(["messages"])
val_dataset = dataset["val"].select_columns(["messages"])

# Response-only loss needs a chat template with {% generation %} markers around the assistant turns. Gemma 3's stock
# template has none: older TRL (e.g. 0.23) then fails with "chat template doesn't generate assistant masks", while newer
# TRL patches it itself. This template renders exactly the same text as Gemma 3's, plus the markers. It is only active
# while the trainer tokenizes the dataset; the original is restored afterwards so Cells 7-8 save an unmodified tokenizer.
GEMMA3_TRAINING_TEMPLATE = """{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- set first_user_prefix = messages[0]['content'] + '\n\n' -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = '' -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}
    {%- endif -%}
    {%- if message['role'] == 'assistant' -%}
        {{ '<start_of_turn>model\n' }}
        {%- generation -%}
            {{ message['content'] | trim ~ '<end_of_turn>\n' }}
        {%- endgeneration -%}
    {%- else -%}
        {{ '<start_of_turn>' ~ message['role'] ~ '\n' ~ (first_user_prefix if loop.first else '') ~ (message['content'] | trim) ~ '<end_of_turn>\n' }}
    {%- endif -%}
{%- endfor -%}
{%- if add_generation_prompt -%}
    {{ '<start_of_turn>model\n' }}
{%- endif -%}"""

ORIGINAL_CHAT_TEMPLATE = tokenizer.chat_template
if not re.search(r"\{%-?\s*generation\s*-?%\}", ORIGINAL_CHAT_TEMPLATE or ""):
    tokenizer.chat_template = GEMMA3_TRAINING_TEMPLATE

BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 3
# warmup_steps must be an int on older transformers (a 0.05 ratio is only accepted by newer ones): 5% of all optimizer steps
steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
WARMUP_STEPS = max(1, math.ceil(0.05 * steps_per_epoch * EPOCHS))

# TRL v0.14+ natively supports assistant_only_loss in SFTConfig for conversational datasets.
# In TRL >= 0.20, DataCollatorForCompletionOnlyLM has been deprecated and removed.
try:
    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=EPOCHS,
        fp16=False,
        bf16=USE_BF16,  # set in cell 4; False on T4 (fp32 compute)
        save_strategy="epoch",
        eval_strategy="epoch",
        max_length=512,
        assistant_only_loss=True,  # Response-only loss: computes loss ONLY on assistant turns
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=training_args,
    )
except TypeError as e:
    # Fallback for legacy TRL versions (< 0.14) that required DataCollatorForCompletionOnlyLM
    try:
        from trl import DataCollatorForCompletionOnlyLM
    except ImportError:
        raise e from None  # not a legacy TRL: show the real error instead of the missing-collator one

    def format_prompts(example):
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

    train_dataset = dataset["train"].map(format_prompts, remove_columns=dataset["train"].column_names)
    val_dataset = dataset["val"].map(format_prompts, remove_columns=dataset["val"].column_names)

    collator = DataCollatorForCompletionOnlyLM(
        response_template="<start_of_turn>model\n",
        instruction_template="<start_of_turn>user\n",
        tokenizer=tokenizer,
    )

    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=EPOCHS,
        fp16=False,
        bf16=USE_BF16,
        save_strategy="epoch",
        eval_strategy="epoch",
        dataset_text_field="text",
        max_length=512,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
        processing_class=tokenizer,
        args=training_args,
    )
finally:
    # The dataset is tokenized by now; keep the saved tokenizer (Cells 7-8) on Gemma's stock template
    tokenizer.chat_template = ORIGINAL_CHAT_TEMPLATE

trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,5.111789,3.461522,3.429781,0.275454,20377.000000
2,3.390819,3.029978,2.934486,0.346929,40754.000000
3,2.688868,2.980380,2.682505,0.359846,61131.000000


TrainOutput(global_step=42, training_loss=3.5005981581551686, metrics={'train_runtime': 188.6095, 'train_samples_per_second': 1.686, 'train_steps_per_second': 0.223, 'total_flos': 299320726740480.0, 'train_loss': 3.5005981581551686, 'epoch': 3.0})

In [14]:
# 6. Test Interactive Multi-Persona Inference
test_messages = [
    {
        "role": "system",
        "content": "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."
    },
    {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, top_p=0.9, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n🤖 SINMATE RESPONSE:", response)



🤖 SINMATE RESPONSE: මහන්, ඔයනේ බෑවෙයි, මගේ චුල් එකේම ගොඩක් ඇස් ගියානේ. ඔයාගේ අයිටුල් එකේ ගියානේද?


In [15]:
# 7. Save LoRA Adapters
model.save_pretrained("sinmate-gemma-3-1b-lora")
tokenizer.save_pretrained("sinmate-gemma-3-1b-lora")
print("Adapters saved successfully to sinmate-gemma-3-1b-lora!")

Adapters saved successfully to sinmate-gemma-3-1b-lora!


In [16]:
# 8. Merge LoRA with Base Model for GGUF Export
import sys
import gc
import json
import torch

# Fix Colab incompatibility where an outdated torchao (< 0.16.0) is pre-installed.
# Bypasses torchao dispatcher check in PEFT so standard PyTorch linear LoRA merge succeeds.
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    if hasattr(peft.import_utils, "TORCHAO_MINIMUM_VERSION"):
        peft.import_utils.TORCHAO_MINIMUM_VERSION = "0.0.0"
    for mod_name, mod in list(sys.modules.items()):
        if mod_name.startswith("peft") and hasattr(mod, "is_torchao_available"):
            setattr(mod, "is_torchao_available", lambda: False)
except Exception:
    pass

from peft import PeftModel
from transformers import AutoModelForCausalLM

# Clear CUDA cache before merging on CPU to avoid memory pressure
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading base model in FP16 to merge LoRA...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
)
merged_model = PeftModel.from_pretrained(base_model, "sinmate-gemma-3-1b-lora")
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("sinmate-gemma-3-1b-merged")
tokenizer.save_pretrained("sinmate-gemma-3-1b-merged")

# Align vocab_size in config.json with actual tokenizer vocabulary to satisfy llama.cpp GGUF converter
config_path = "sinmate-gemma-3-1b-merged/config.json"
try:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    max_tok_id = max(tokenizer.get_vocab().values())
    if cfg.get("vocab_size", 0) <= max_tok_id:
        cfg["vocab_size"] = max_tok_id + 1
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print(f"Aligned config.json vocab_size -> {cfg['vocab_size']} (max token ID: {max_tok_id})")
except Exception as e:
    print(f"Note: vocab_size alignment: {e}")

print("Merged model saved successfully to ./sinmate-gemma-3-1b-merged!")


Loading base model in FP16 to merge LoRA...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Aligned config.json vocab_size -> 262145 (max token ID: 262144)
Merged model saved successfully to ./sinmate-gemma-3-1b-merged!


In [17]:
# 9. Convert Merged Model to 4-bit GGUF via llama.cpp
import os
import sys
import json

# Helper to locate directories regardless of Colab working directory (/content vs /content/Sinmate)
def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("/content", target_name),
        os.path.join("/content/Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

# 1. Locate merged model directory
merged_model_dir = resolve_path("sinmate-gemma-3-1b-merged")
if not merged_model_dir:
    raise FileNotFoundError("Could not find 'sinmate-gemma-3-1b-merged'. Please ensure Cell 8 completed successfully.")
print(f"Found merged model directory: {merged_model_dir}")

# 2. Clone llama.cpp and install conversion requirements (avoiding legacy protobuf downgrade)
llama_cpp_dir = resolve_path("llama.cpp")
if not llama_cpp_dir:
    !git clone https://github.com/ggml-org/llama.cpp.git
    llama_cpp_dir = resolve_path("llama.cpp") or os.path.abspath("llama.cpp")
print(f"Found llama.cpp directory: {llama_cpp_dir}")

req_convert = os.path.join(llama_cpp_dir, "requirements", "requirements-convert_hf_to_gguf.txt")
if os.path.exists(req_convert):
    !pip install -q -r "{req_convert}"
else:
    !pip install -q -r "{llama_cpp_dir}/requirements.txt"
!pip install -q --upgrade "protobuf>=5.0.0"

# 3. Gemma 3 must be converted from its SentencePiece `tokenizer.model`. Without it llama.cpp falls back to the
#    GPT-2/BPE route, mis-detects the pre-tokenizer and writes 262145 tokens (incl. <image_soft_token>) against
#    262144 embedding rows - a GGUF that Ollama cannot load (HTTP 500 on every chat request).
tok_model_path = os.path.join(merged_model_dir, "tokenizer.model")
if not os.path.exists(tok_model_path):
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download("google/gemma-3-1b-it", "tokenizer.model", local_dir=merged_model_dir)
    except Exception as e:
        print(f"⚠️ Could not fetch tokenizer.model from google/gemma-3-1b-it: {e}")
has_spm = os.path.exists(tok_model_path)
print(f"SentencePiece tokenizer.model present: {has_spm}")

# 4. config.json vocab_size must equal the embedding rows of the weights. Earlier runs bumped it to 262145 (the
#    tokenizer's extra <image_soft_token>) and that value persists on disk: the converter then pads the vocab to
#    262145 tokens against 262144 embedding rows. Reset it from the actual weights.
if has_spm:
    import glob
    from safetensors import safe_open
    config_path = os.path.join(merged_model_dir, "config.json")
    embd_rows = None
    for st_file in sorted(glob.glob(os.path.join(merged_model_dir, "*.safetensors"))):
        with safe_open(st_file, framework="pt") as f:
            for key in f.keys():
                if key.endswith("embed_tokens.weight"):
                    embd_rows = f.get_slice(key).get_shape()[0]
                    break
        if embd_rows:
            break
    if embd_rows and os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        if cfg.get("vocab_size") != embd_rows:
            print(f"Reset config.json vocab_size {cfg.get('vocab_size')} -> {embd_rows} (embedding rows)")
            cfg["vocab_size"] = embd_rows
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
        else:
            print(f"config.json vocab_size already matches embedding rows ({embd_rows})")

# 5. Only without tokenizer.model: fall back to the vocab_size / assertion workarounds for the BPE route
if not has_spm:
    config_path = os.path.join(merged_model_dir, "config.json")
    if os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(merged_model_dir)
        max_tok_id = max(tok.get_vocab().values())
        if cfg.get("vocab_size", 0) <= max_tok_id:
            cfg["vocab_size"] = max_tok_id + 1
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
            print(f"Aligned config.json vocab_size -> {cfg['vocab_size']} (max token ID: {max_tok_id})")
    # Patch conversion/base.py in llama.cpp to prevent assertion crashes
    base_py = os.path.join(llama_cpp_dir, "conversion", "base.py")
    if os.path.exists(base_py):
        with open(base_py, "r", encoding="utf-8") as f:
            content = f.read()
        if "assert max(tokenizer.vocab.values()) < vocab_size" in content:
            content = content.replace(
                "assert max(tokenizer.vocab.values()) < vocab_size",
                "vocab_size = max(vocab_size, max(tokenizer.vocab.values()) + 1)"
            )
            with open(base_py, "w", encoding="utf-8") as f:
                f.write(content)
            print("Patched llama.cpp conversion/base.py assertion check")

# 6. Convert HuggingFace checkpoint to FP16 GGUF
f16_gguf = os.path.abspath("sinmate-gemma-3-1b-f16.gguf")
q4_gguf = os.path.abspath("sinmate-gemma-3-1b-q4_k_m.gguf")
# Remove GGUFs from earlier runs so a failed conversion can never leave a stale file in use
for stale in (f16_gguf, q4_gguf):
    if os.path.exists(stale):
        os.remove(stale)
convert_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
!python "{convert_script}" "{merged_model_dir}" --outfile "{f16_gguf}" --outtype f16

# 7. Locate or compile llama-quantize with memory-safe parallelism (-j 2)
quantize_bin = None
for bin_candidate in [
    os.path.join(llama_cpp_dir, "build", "bin", "llama-quantize"),
    os.path.join(llama_cpp_dir, "build", "llama-quantize"),
    os.path.join(llama_cpp_dir, "llama-quantize"),
    "/content/llama.cpp/build/bin/llama-quantize",
    "/content/Sinmate/llama.cpp/build/bin/llama-quantize",
]:
    if os.path.exists(bin_candidate):
        quantize_bin = os.path.abspath(bin_candidate)
        break

if not quantize_bin:
    !cmake -B "{llama_cpp_dir}/build" "{llama_cpp_dir}" -DLLAMA_BUILD_SERVER=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
    !cmake --build "{llama_cpp_dir}/build" --config Release -j 2 --target llama-quantize
    quantize_bin = os.path.join(llama_cpp_dir, "build", "bin", "llama-quantize")

print(f"Using llama-quantize binary: {quantize_bin}")

# 8. Quantize to Q4_K_M (4-bit)
if os.path.exists(f16_gguf):
    !chmod +x "{quantize_bin}"
    !"{quantize_bin}" "{f16_gguf}" "{q4_gguf}" Q4_K_M
else:
    print(f"⚠️ {f16_gguf} was not found. Please review convert_hf_to_gguf logs above.")

if os.path.exists(q4_gguf):
    size_mb = os.path.getsize(q4_gguf) / (1024 * 1024)
    print(f"\n🎉 GGUF quantization complete: {q4_gguf} ({size_mb:.1f} MB)")
else:
    print("\n⚠️ Quantization output file not found. Please check logs above.")


Found merged model directory: /content/Sinmate/sinmate-gemma-3-1b-merged
Cloning into 'llama.cpp'...
remote: Enumerating objects: 146488, done.
remote: Counting objects: 100% (693/693), done.
remote: Compressing objects: 100% (368/368), done.
remote: Total 146488 (delta 489), reused 325 (delta 325), pack-reused 145795 (from 3)
Receiving objects: 100% (146488/146488), 450.35 MiB | 19.00 MiB/s, done.
Resolving deltas: 100% (106172/106172), done.
Updating files: 100% (3930/3930), done.
Found llama.cpp directory: /content/Sinmate/llama.cpp
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires protobuf!=5.28.0,!=5.29.0,<8,>=5, but you have protobuf 4.25.9 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33

tokenizer.model: reconstructing file:   0%|          |  0.00B / 4.69MB            

tokenizer.model: downloading bytes:           |  0.00B            

SentencePiece tokenizer.model present: True
Reset config.json vocab_size 262145 -> 262144 (embedding rows)
INFO:hf-to-gguf:Loading model: sinmate-gemma-3-1b-merged
INFO:hf-to-gguf:Model architecture: Gemma3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {1152, 262144}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.float16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.float16 --> F16, shape = {6912, 1152}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.float16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.float16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.float16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.post_ffw_norm.weight,        torch

In [18]:
# 10. Generate Modelfile for Ollama
import os

def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("/content", target_name),
        os.path.join("/content/Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

# Locate the quantized GGUF model produced by Cell 9
gguf_path = resolve_path("sinmate-gemma-3-1b-q4_k_m.gguf") or os.path.abspath("sinmate-gemma-3-1b-q4_k_m.gguf")
print(f"Target GGUF file: {gguf_path}")

# Gemma has no system role: the HF chat template used for fine-tuning folds the system prompt
# into the first user turn, so the Ollama TEMPLATE must do the same to match training.
modelfile_template = """FROM {GGUF_PATH}
TEMPLATE \"\"\"<start_of_turn>user
{{ if .System }}{{ .System }}

{{ end }}{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
\"\"\"
PARAMETER stop "<start_of_turn>"
PARAMETER stop "<end_of_turn>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
"""

modelfile = modelfile_template.replace("{GGUF_PATH}", gguf_path)

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

print("\n✅ Modelfile successfully generated at:", os.path.abspath("Modelfile"))
print(f"Points to GGUF: {gguf_path}")
print("\n👉 Next Step: Run Cell 11 to install Ollama, start the daemon, and build 'sinmate-gemma3:1b' directly on Colab!")

Target GGUF file: /content/Sinmate/sinmate-gemma-3-1b-q4_k_m.gguf

✅ Modelfile successfully generated at: /content/Sinmate/Modelfile
Points to GGUF: /content/Sinmate/sinmate-gemma-3-1b-q4_k_m.gguf

👉 Next Step: Run Cell 11 to install Ollama, start the daemon, and build 'sinmate-gemma3:1b' directly on Colab!


In [19]:
# 11. Setup Ollama & Build Sinmate Model in Colab
import os
import gc
import json
import time
import shutil
import subprocess
import urllib.error
import urllib.request

import torch

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL_TAG = "sinmate-gemma3:1b"

def resolve_path(target_name):
    candidates = [
        target_name,
        os.path.join("Sinmate", target_name),
        os.path.join("/content", target_name),
        os.path.join("/content/Sinmate", target_name),
        os.path.join("..", target_name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

def ollama_up():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False

def ollama_error(e, log_lines=25):
    """Readable reason for a failed Ollama request: the response body (urllib hides it) + the server log tail."""
    body = ""
    if isinstance(e, urllib.error.HTTPError):
        try:
            body = e.read().decode("utf-8", "replace")
        except Exception:
            pass
    tail = ""
    if os.path.exists("ollama_server.log"):
        with open("ollama_server.log", "r", errors="replace") as f:
            tail = "".join(f.readlines()[-log_lines:])
    return f"{e}\n{body}\n--- ollama_server.log (tail) ---\n{tail}"

def check_gguf_vocab(path):
    """Fail fast if the GGUF has more tokens than embedding rows - Ollama then answers every chat with HTTP 500."""
    try:
        from gguf import GGUFReader
        reader = GGUFReader(path)
        n_tokens = len(reader.fields["tokenizer.ggml.tokens"].data)
        embd_rows = next(int(t.shape[1]) for t in reader.tensors if t.name == "token_embd.weight")
    except (ImportError, KeyError, StopIteration) as e:
        print(f"ℹ️ Skipping GGUF vocab check ({type(e).__name__}: {e})")
        return
    print(f"GGUF vocab check: {n_tokens} tokens vs {embd_rows} embedding rows")
    if n_tokens != embd_rows:
        raise RuntimeError(
            f"❌ GGUF is broken: {n_tokens} tokens but {embd_rows} embedding rows. "
            "Re-run Cell 9 (it now converts with Gemma's SentencePiece tokenizer.model), then Cell 10 and this cell."
        )

# 1. Clean up unused PyTorch tensors to leave maximum free VRAM for Ollama
for var_name in ["model", "trainer", "base_model", "merged_model"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Install Ollama on the Colab VM (its installer extracts a .tar.zst, so zstd must exist first)
if shutil.which("ollama") is None:
    print("📦 Installing zstd + Ollama on Colab VM...")
    !apt-get install -y -qq zstd pciutils
    !curl -fsSL https://ollama.com/install.sh | sh
    if shutil.which("ollama") is None:
        raise RuntimeError("❌ Ollama installation failed - see the installer output above.")
else:
    print("✅ Ollama binary is already installed.")

# 3. Start background Ollama daemon if not running
if ollama_up():
    print("✅ Ollama server is already running.")
else:
    print("🚀 Starting Ollama daemon in background...")
    log_fp = open("ollama_server.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=log_fp, stderr=subprocess.STDOUT, start_new_session=True)
    for attempt in range(30):
        if ollama_up():
            print(f"✅ Ollama server is live at {OLLAMA_URL} (attempt {attempt + 1})")
            break
        time.sleep(1)
    else:
        with open("ollama_server.log", "r") as f:
            print("".join(f.readlines()[-20:]))
        raise RuntimeError("❌ Ollama server did not respond - see log output above.")

# 4. Locate Modelfile and GGUF
modelfile_path = resolve_path("Modelfile")
gguf_path = resolve_path("sinmate-gemma-3-1b-q4_k_m.gguf")

if not modelfile_path:
    raise FileNotFoundError("❌ 'Modelfile' not found. Please run Cell 10 first!")
if not gguf_path:
    raise FileNotFoundError("❌ 'sinmate-gemma-3-1b-q4_k_m.gguf' not found. Please run Cell 9 first!")

print(f"Modelfile: {modelfile_path}")
print(f"GGUF: {gguf_path}")
check_gguf_vocab(gguf_path)

# Ensure Modelfile points to the exact absolute GGUF path
with open(modelfile_path, "r", encoding="utf-8") as f:
    mf_lines = f.read().splitlines()

mf_lines = [f"FROM {gguf_path}" if line.startswith("FROM ") else line for line in mf_lines]

with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write("\n".join(mf_lines) + "\n")

# 5. Build the Sinmate model in Ollama
print(f"\n🔨 Building Ollama model '{MODEL_TAG}' from Modelfile...")
!ollama create {MODEL_TAG} -f "{modelfile_path}"

# 6. List Ollama models
print("\n📋 Available Ollama models:")
!ollama list

# 7. Quick verification through the HTTP API (the `ollama run` CLI prints spinner escape codes into the cell output)
print("\n🧪 Running quick verification prompt via Ollama API...")
verify_payload = json.dumps({
    "model": MODEL_TAG,
    "stream": False,
    "messages": [
        {"role": "system", "content": "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."},
        {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"},
    ],
    "options": {"temperature": 0.7, "top_p": 0.9, "num_predict": 64},
}).encode("utf-8")
verify_req = urllib.request.Request(OLLAMA_URL + "/api/chat", data=verify_payload, headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(verify_req, timeout=300) as resp:
        print("🤖 SINMATE RESPONSE:", json.loads(resp.read().decode("utf-8"))["message"]["content"])
except Exception as e:
    print(f"⚠️ Verification request failed: {ollama_error(e)}")

📦 Installing zstd + Ollama on Colab VM...
Selecting previously unselected package pci.ids.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.10.0-2build1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Setting up pciutils (1:3.10.0-2build1) ...
Processing triggers for man-db (2.12.0-4build2) ...

In [20]:
# 12. Real-Time Interactive Multi-Persona Chat
import os
import sys
import json
import urllib.error
import urllib.request

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL_TAG = "sinmate-gemma3:1b"
MAX_HISTORY = 6  # last N messages kept as conversation context

# 1. Define Persona Prompts (Kathitha Bhashawa / Sri Lankan Spoken Sinhala)
PERSONAS = {
    "1": ("BEST_FRIEND", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."),
    "2": ("ROMANTIC_PARTNER", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: ROMANTIC_PARTNER]\nYou are the user's loving, affectionate romantic partner. Responses must strictly be 1-2 warm, spoken colloquial sentences."),
    "3": ("COUNSELLOR", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: COUNSELLOR]\nYou are a calm, deeply empathetic listener. Responses must strictly be 1-2 grounded, supportive spoken colloquial sentences."),
    "4": ("PLAYFUL_SIBLING", "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: PLAYFUL_SIBLING]\nYou are the user's witty, teasing sibling. Responses must strictly be 1-2 playful, humorous spoken colloquial sentences."),
}

# 2. Check if the Ollama engine is up and has the Sinmate model
def is_ollama_ready():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=2) as r:
            data = json.loads(r.read().decode("utf-8"))
            return any("sinmate-gemma3" in m.get("name", "") for m in data.get("models", []))
    except Exception:
        return False

ollama_available = is_ollama_ready()

def ollama_error(e, log_lines=25):
    """Readable reason for a failed Ollama request: the response body (urllib hides it) + the server log tail."""
    body = ""
    if isinstance(e, urllib.error.HTTPError):
        try:
            body = e.read().decode("utf-8", "replace")
        except Exception:
            pass
    tail = ""
    if os.path.exists("ollama_server.log"):
        with open("ollama_server.log", "r", errors="replace") as f:
            tail = "".join(f.readlines()[-log_lines:])
    return f"{e}\n{body}\n--- ollama_server.log (tail) ---\n{tail}"

def build_messages(user_text, persona_key, history):
    _, system_prompt = PERSONAS.get(persona_key, PERSONAS["1"])
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        messages.extend(history[-MAX_HISTORY:])
    messages.append({"role": "user", "content": user_text})
    return messages

# 3. Streaming chat functions (both return the full reply so it can be kept as history)
def chat_via_ollama(user_text, persona_key="1", history=None):
    persona_name = PERSONAS.get(persona_key, PERSONAS["1"])[0]
    payload = json.dumps({
        "model": MODEL_TAG,
        "messages": build_messages(user_text, persona_key, history),
        "stream": True,
        "options": {
            "temperature": 0.7,
            "top_p": 0.9,
            "num_predict": 128,
            "stop": ["<start_of_turn>", "<end_of_turn>"],
        },
    }).encode("utf-8")

    req = urllib.request.Request(
        OLLAMA_URL + "/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
    )

    print(f"\n[{persona_name}] Sinmate: ", end="")
    full_response = []
    with urllib.request.urlopen(req, timeout=300) as resp:
        for line in resp:
            if not line.strip():
                continue
            token = json.loads(line.decode("utf-8")).get("message", {}).get("content", "")
            if token:
                sys.stdout.write(token)
                sys.stdout.flush()
                full_response.append(token)
    print()
    return "".join(full_response)

# Hugging Face fallback if Ollama is not running
hf_model = None
hf_tokenizer = None

def init_hf_fallback():
    global hf_model, hf_tokenizer
    if hf_model is not None:
        return

    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["USE_TF"] = "0"
    try:
        import transformers.utils.import_utils as tr_import
        tr_import.is_tf_available = lambda: False
        tr_import.is_flax_available = lambda: False
        tr_import.is_torch_available = lambda: True
    except Exception:
        pass

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_dirs = [
        "sinmate-gemma-3-1b-merged",
        "Sinmate/sinmate-gemma-3-1b-merged",
        "/content/Sinmate/sinmate-gemma-3-1b-merged",
        "/content/sinmate-gemma-3-1b-merged",
    ]
    m_path = "google/gemma-3-1b-it"
    for p in model_dirs:
        if os.path.exists(p):
            m_path = os.path.abspath(p)
            break

    # Gemma 3 overflows in fp16 (see Cell 4): bf16 on Ampere+, fp32 on the T4
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32

    print(f"Loading HuggingFace fallback from: {m_path} ...")
    hf_tokenizer = AutoTokenizer.from_pretrained(m_path)
    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token
    hf_model = AutoModelForCausalLM.from_pretrained(m_path, dtype=dtype, device_map="auto")
    hf_model.eval()

def chat_via_hf(user_text, persona_key="1", history=None):
    init_hf_fallback()
    import torch
    from transformers import TextStreamer

    persona_name = PERSONAS.get(persona_key, PERSONAS["1"])[0]
    messages = build_messages(user_text, persona_key, history)

    prompt = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = hf_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(hf_model.device)
    streamer = TextStreamer(hf_tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"\n[{persona_name}] Sinmate: ", end="")
    with torch.no_grad():
        outputs = hf_model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=64,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    return hf_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def chat_sinmate(user_text, persona_key="1", history=None):
    global ollama_available
    if ollama_available:
        try:
            return chat_via_ollama(user_text, persona_key, history)
        except urllib.error.URLError as e:  # HTTPError is a subclass
            print(f"\n⚠️ Ollama request failed - falling back to HuggingFace Transformers:\n{ollama_error(e)}")
            ollama_available = False
    return chat_via_hf(user_text, persona_key, history)

# 4. Print status & run automated multi-persona tests
print("=" * 60)
if ollama_available:
    print(f"🚀 SINMATE REAL-TIME CHAT (ENGINE: Ollama - {MODEL_TAG})")
else:
    print("🚀 SINMATE REAL-TIME CHAT (ENGINE: HuggingFace Transformers)")
print("=" * 60)

sample_tests = [
    ("Mata ada mara kammali ban, monada karanna hoda?", "1"),
    ("Mata ada godak mahansi darling.", "2"),
    ("Mata godak bayai exam eka gana.", "3"),
    ("Mage chocolate eka kawe oyada?", "4"),
]

for prompt_text, p_key in sample_tests:
    print(f"\nUser ({PERSONAS[p_key][0]}): {prompt_text}")
    chat_sinmate(prompt_text, persona_key=p_key)

print("\n" + "=" * 60)
print("💬 Interactive Real-Time Chat:")
print("Commands: 'p 1-4' to change persona | 'exit' to quit")
print("Personas: [1] Best Friend   [2] Romantic Partner   [3] Counsellor   [4] Playful Sibling")
print("=" * 60)

try:
    current_persona = "1"
    history = []
    while True:
        user_input = input("\nYou (or 'p 1-4' to switch, 'exit' to quit): ").strip()
        if not user_input or user_input.lower() in ["exit", "quit"]:
            print("Chat session ended. Sinmate signing off!")
            break
        if user_input.startswith("p ") and user_input[2:] in PERSONAS:
            current_persona = user_input[2:]
            history = []  # new persona starts a fresh conversation
            print(f"Switched persona to: {PERSONAS[current_persona][0]}")
            continue
        reply = chat_sinmate(user_input, persona_key=current_persona, history=history)
        history += [{"role": "user", "content": user_input}, {"role": "assistant", "content": reply}]
except (EOFError, KeyboardInterrupt):
    print("\nSession finished.")

🚀 SINMATE REAL-TIME CHAT (ENGINE: Ollama - sinmate-gemma3:1b)

User (BEST_FRIEND): Mata ada mara kammali ban, monada karanna hoda?

[BEST_FRIEND] Sinmate: අම්මතට කෑගනියෝ උඹගනිය වෙලානේ, ආදරෙන් කෑගනියෝ. දැන් ඇය දාටුනේ බෑබෑ වෙලා ඉඳලා ලදී.

User (ROMANTIC_PARTNER): Mata ada godak mahansi darling.

[ROMANTIC_PARTNER] Sinmate: හොඳක්ම ආදරෙන් කවදුවෙක්ම වෙන්න මගේ රසවත් එකෙක්යි.

User (COUNSELLOR): Mata godak bayai exam eka gana.

[COUNSELLOR] Sinmate: මගේ බෑදන්, කතා කරන්නේ මගේ හෙමින්. ඔයාගේ කම්පෙල ගොඩක් යම් සර්ස් එකක් ඇල්ලෙනවා, දැන් ඇත්තේ මගේ අදහස් එකක් ගොඩක් ළඟ කියලා.

User (PLAYFUL_SIBLING): Mage chocolate eka kawe oyada?

[PLAYFUL_SIBLING] Sinmate: ඔව්, එකට ගුඩුවක් එකම ඇස් වෙලා තියෙනවා, ඒ වෙලා මගේ සතුටේ මගේ සෙදරේයි.

💬 Interactive Real-Time Chat:
Commands: 'p 1-4' to change persona | 'exit' to quit
Personas: [1] Best Friend   [2] Romantic Partner   [3] Counsellor   [4] Playful Sibling

[BEST_FRIEND] Sinmate: ආදරෙයි බං, වැඩේ නෙයි මගේ රසවත් වතුර එකට දාලා ලියන්න එපා නෙයි! දැන් මට ඔයාගේ පැටියෙක්